# 1D Unit Random Walk: Return-to-Origin Simulation

This notebook simulates the classic one-dimensional unit random walk.

Model:
- Start at x0 = 0
- At each step, move +1 or -1 with equal probability
- Stop immediately when the walk returns to x = 0 after leaving it

Outputs are saved in `outputs/unit_walk_1D/`.

In [1]:
from pathlib import Path
import json
import math
from typing import Dict, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Configuration
RANDOM_SEED = 20260407
N_TRIALS = 300
MAX_STEPS = 20_000

NOTEBOOK_NAME = 'unit_walk_1D.ipynb'

cwd = Path.cwd()
if (cwd / NOTEBOOK_NAME).exists():
    NOTEBOOK_DIR = cwd
elif (cwd / 'random_walks' / NOTEBOOK_NAME).exists():
    NOTEBOOK_DIR = cwd / 'random_walks'
else:
    NOTEBOOK_DIR = cwd

OUTPUT_DIR = NOTEBOOK_DIR / 'outputs' / 'unit_walk_1D'
PATHS_DIR = OUTPUT_DIR / 'paths'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PATHS_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')
rng = np.random.default_rng(RANDOM_SEED)

print(f'Notebook directory: {NOTEBOOK_DIR}')
print(f'Output directory:   {OUTPUT_DIR}')

Notebook directory: /home/manpazito/projects/fun_mini_sims/random_walks
Output directory:   /home/manpazito/projects/fun_mini_sims/random_walks/outputs/unit_walk_1D


## Mathematical Description

Let xi_k be independent steps with P(xi_k = +1) = P(xi_k = -1) = 1/2.

Position after n steps:
`X_n = sum_{k=1..n} xi_k`, with `X_0 = 0`.

First return time used here:
`T = min{ n >= 1 : X_n = 0 }`.

Return times can be long, so each trial uses a `MAX_STEPS` safeguard.
Trials that do not return by then are marked as censored.

In [2]:
def simulate_unit_walk_1d(rng: np.random.Generator, max_steps: int) -> Dict[str, object]:
    # Simulate one 1D walk until first return or max_steps.
    position = 0
    positions: List[int] = [position]

    min_position = 0
    max_position = 0
    max_abs_displacement = 0
    returned_to_origin = False

    for _ in range(max_steps):
        step = int(rng.choice((-1, 1)))
        position += step
        positions.append(position)

        min_position = min(min_position, position)
        max_position = max(max_position, position)
        max_abs_displacement = max(max_abs_displacement, abs(position))

        if position == 0:
            returned_to_origin = True
            break

    steps_until_stop = len(positions) - 1
    return_time = float(steps_until_stop) if returned_to_origin else np.nan

    return {
        'positions': positions,
        'steps_until_stop': steps_until_stop,
        'returned_to_origin': returned_to_origin,
        'censored': not returned_to_origin,
        'return_time': return_time,
        'max_abs_displacement': float(max_abs_displacement),
        'min_position': int(min_position),
        'max_position': int(max_position),
        'final_position': int(position),
    }


def save_path_json(path_positions: List[int], trial_index: int, paths_dir: Path) -> Path:
    payload = {
        'trial_index': int(trial_index),
        'positions': [int(v) for v in path_positions],
    }
    out_path = paths_dir / f'trial_{trial_index:04d}.json'
    with out_path.open('w', encoding='utf-8') as f:
        json.dump(payload, f)
    return out_path


def plot_single_walk(positions: List[int], out_path: Path) -> None:
    steps = np.arange(len(positions))
    fig, ax = plt.subplots(figsize=(10, 4.8))
    ax.plot(steps, positions, color='tab:blue', linewidth=2)
    ax.axhline(0, color='black', linewidth=1, alpha=0.8)
    ax.scatter([0], [0], color='green', s=70, marker='o', label='origin')
    ax.scatter([steps[-1]], [positions[-1]], color='red', s=80, marker='X', label='stop point')
    ax.set_title('Single 1D Unit Walk (Position vs Step)')
    ax.set_xlabel('Step number')
    ax.set_ylabel('Position')
    ax.legend(loc='best')
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def plot_return_histogram(summary_df: pd.DataFrame, out_path: Path) -> None:
    completed = summary_df.loc[~summary_df['censored'], 'return_time']
    fig, ax = plt.subplots(figsize=(8, 4.8))
    ax.hist(completed, bins=30, color='tab:blue', edgecolor='white')
    ax.set_title('Return Time Histogram (Completed Trials)')
    ax.set_xlabel('Steps until first return')
    ax.set_ylabel('Count')
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def plot_max_distance_histogram(summary_df: pd.DataFrame, out_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(8, 4.8))
    ax.hist(summary_df['max_abs_displacement'], bins=30, color='tab:orange', edgecolor='white')
    ax.set_title('Max Absolute Displacement Histogram')
    ax.set_xlabel('Max |position| during walk')
    ax.set_ylabel('Count')
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def plot_overlay_walks(all_paths: List[List[int]], out_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(10, 5))
    for path in all_paths:
        steps = np.arange(len(path))
        ax.plot(steps, path, color='tab:blue', alpha=0.12, linewidth=0.8)

    ax.axhline(0, color='black', linewidth=1)
    ax.set_title('Overlay of All 1D Walks (Position vs Step)')
    ax.set_xlabel('Step number')
    ax.set_ylabel('Position')
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)

## Single Example Walk

Simulate one walk from the origin and stop at first return to x = 0.

In [3]:
single_walk = simulate_unit_walk_1d(rng=rng, max_steps=MAX_STEPS)

single_walk_plot_path = OUTPUT_DIR / 'single_walk.png'
plot_single_walk(single_walk['positions'], single_walk_plot_path)

print('Single walk summary:')
print(f"  steps_until_stop:      {single_walk['steps_until_stop']}")
print(f"  returned_to_origin:    {single_walk['returned_to_origin']}")
print(f"  max_abs_displacement:  {single_walk['max_abs_displacement']}")
print(f'  saved plot:            {single_walk_plot_path}')

Single walk summary:
  steps_until_stop:      2
  returned_to_origin:    True
  max_abs_displacement:  1.0
  saved plot:            /home/manpazito/projects/fun_mini_sims/random_walks/outputs/unit_walk_1D/single_walk.png


## Monte Carlo Simulation

For each trial we save summary metrics and a raw path JSON file.

In [4]:
trial_records: List[Dict[str, object]] = []
all_paths: List[List[int]] = []

for trial_idx in range(N_TRIALS):
    walk = simulate_unit_walk_1d(rng=rng, max_steps=MAX_STEPS)
    all_paths.append(walk['positions'])
    save_path_json(walk['positions'], trial_idx, PATHS_DIR)

    trial_records.append(
        {
            'trial_index': trial_idx,
            'steps_until_stop': walk['steps_until_stop'],
            'return_time': walk['return_time'],
            'returned_to_origin': walk['returned_to_origin'],
            'censored': walk['censored'],
            'max_abs_displacement': walk['max_abs_displacement'],
            'min_position': walk['min_position'],
            'max_position': walk['max_position'],
            'final_position': walk['final_position'],
        }
    )

summary_df = pd.DataFrame(trial_records)
summary_csv_path = OUTPUT_DIR / 'summary.csv'
summary_df.to_csv(summary_csv_path, index=False)

completed = summary_df[~summary_df['censored']]

empirical_mean_return_time = float(completed['return_time'].mean()) if len(completed) else np.nan
empirical_variance_return_time = float(completed['return_time'].var(ddof=1)) if len(completed) > 1 else np.nan
empirical_mean_max_abs_displacement = float(summary_df['max_abs_displacement'].mean())

print(f'Trials run:                     {N_TRIALS}')
print(f'Completed returns:              {len(completed)}')
print(f'Censored trials:                {int(summary_df["censored"].sum())}')
print(f'Empirical mean return time:     {empirical_mean_return_time:.3f}')
print(f'Empirical variance return time: {empirical_variance_return_time:.3f}')
print(f'Empirical mean max |x|:         {empirical_mean_max_abs_displacement:.3f}')
print(f'Saved summary CSV:              {summary_csv_path}')

summary_df.head()

Trials run:                     300
Completed returns:              297
Censored trials:                3
Empirical mean return time:     128.572
Empirical variance return time: 554070.496
Empirical mean max |x|:         8.193
Saved summary CSV:              /home/manpazito/projects/fun_mini_sims/random_walks/outputs/unit_walk_1D/summary.csv


,trial_index,steps_until_stop,return_time,returned_to_origin,censored,max_abs_displacement,min_position,max_position,final_position
0,0,22,22.0,True,False,5.0,0,5,0
1,1,24,24.0,True,False,5.0,-5,0,0
2,2,20,20.0,True,False,5.0,0,5,0
3,3,8622,8622.0,True,False,156.0,0,156,0
4,4,2,2.0,True,False,1.0,0,1,0


## Visualization of Monte Carlo Results

We plot histograms of return times and max displacement.

In [5]:
return_hist_path = OUTPUT_DIR / 'return_time_histogram.png'
max_dist_hist_path = OUTPUT_DIR / 'max_distance_histogram.png'

plot_return_histogram(summary_df, return_hist_path)
plot_max_distance_histogram(summary_df, max_dist_hist_path)

print(f'Saved: {return_hist_path}')
print(f'Saved: {max_dist_hist_path}')

Saved: /home/manpazito/projects/fun_mini_sims/random_walks/outputs/unit_walk_1D/return_time_histogram.png
Saved: /home/manpazito/projects/fun_mini_sims/random_walks/outputs/unit_walk_1D/max_distance_histogram.png


## Final Overlay Plot

Overlay all Monte Carlo trajectories on a single position-vs-step chart.

In [6]:
overlay_path = OUTPUT_DIR / 'overlay_walks.png'
plot_overlay_walks(all_paths, overlay_path)
print(f'Saved: {overlay_path}')

Saved: /home/manpazito/projects/fun_mini_sims/random_walks/outputs/unit_walk_1D/overlay_walks.png
